In [ ]:
import os
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Resolve absolute paths to local Silver Delta table directories
PROJECT_ROOT = os.path.dirname(os.getcwd())


GOLD_PATH = os.path.join(PROJECT_ROOT, "data", "tables", "gold_security_alerts")

# Initialize SparkSession with Delta support
builder = (
    SparkSession.builder.appName("Query_Gold")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [ ]:
# Read the materialized Gold summary table
gold_alerts_df = spark.read.format("delta").load(GOLD_PATH)

print(f"Total Operational Security Alerts Triggered: {gold_alerts_df.count()}")
gold_alerts_df.orderBy(col("_alert_triggered_at").desc()).show(10, truncate=False)